# Intent Classifier Training & Comparison — BERT vs DistilBERT vs RoBERTa

Fine-tunes and compares three transformer encoders on the **intent classification** task (Dataset A: `text -> intent`) for the Educational Chatbot.

**Run on Google Colab / Kaggle with a GPU** (Runtime → Change runtime type → GPU). Your local MX450 (2 GB) is not used.

### Steps
1. Install dependencies
2. Upload `Dataset_A_SRKI.csv` (or `SU_final_250k_A.csv`)
3. Train `bert-base-uncased`, `distilbert-base-uncased`, `roberta-base`
4. Compare accuracy / precision / recall / F1
5. Save the best model and optionally push it to the Hugging Face Hub.

In [ ]:
!pip -q install "transformers>=4.38" "datasets>=2.16" "accelerate>=0.27" evaluate scikit-learn pandas matplotlib seaborn huggingface-hub

In [ ]:
# ---- Configuration ----
DATA_CSV = "Dataset_A_SRKI.csv"   # upload this file (or SU_final_250k_A.csv)
INSTITUTION = "srki"              # 'srki' or 'su'
MODELS = {
    "bert": "bert-base-uncased",
    "distilbert": "distilbert-base-uncased",
    "roberta": "roberta-base",
}
NUM_EPOCHS = 3
MAX_LEN = 128
BATCH = 32          # lower to 16 if you hit out-of-memory
# For the huge SU dataset (250k) you can subsample for a faster first run:
MAX_ROWS = None     # e.g. 60000
PUSH_TO_HUB = False
HF_USER = "your-hf-username"

In [ ]:
from google.colab import files
import os
if not os.path.exists(DATA_CSV):
    print('Upload', DATA_CSV)
    files.upload()

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

df = pd.read_csv(DATA_CSV, dtype=str, keep_default_na=False)
df.columns = [c.strip().lower() for c in df.columns]
df = df[(df['text'].str.len() > 0) & (df['intent'].str.len() > 0)].drop_duplicates('text')
if MAX_ROWS:
    df = df.groupby('intent', group_keys=False).apply(lambda g: g.sample(min(len(g), MAX_ROWS // df['intent'].nunique()), random_state=42))
labels = sorted(df['intent'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df['label'] = df['intent'].map(label2id)
print(len(df), 'rows,', len(labels), 'intents')

from collections import Counter
def safe_split(frame, test_size):
    strat = frame['label'] if min(Counter(frame['label']).values()) >= 2 else None
    return train_test_split(frame, test_size=test_size, random_state=42, stratify=strat)
train_df, temp_df = safe_split(df, 0.2)
val_df, test_df = safe_split(temp_df, 0.5)
print('train', len(train_df), 'val', len(val_df), 'test', len(test_df))

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

def make_ds(frame):
    return Dataset.from_dict({'text': frame['text'].tolist(), 'label': frame['label'].tolist()})

raw = {k: make_ds(v) for k, v in [('train', train_df), ('val', val_df), ('test', test_df)]}

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted', zero_division=0)
    return {'accuracy': accuracy_score(p.label_ids, preds), 'precision': pr, 'recall': rc, 'f1': f1}

def train_one(name, checkpoint):
    print('\n==============', name, checkpoint, '==============')
    tok = AutoTokenizer.from_pretrained(checkpoint)
    def tk(b): return tok(b['text'], truncation=True, max_length=MAX_LEN)
    ds = {k: v.map(tk, batched=True) for k, v in raw.items()}
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(labels), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f'out_{name}', num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        learning_rate=2e-5, weight_decay=0.01, eval_strategy='epoch',
        save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1',
        fp16=torch.cuda.is_available(), logging_steps=200, report_to='none')
    trainer = Trainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
                      tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    trainer.train()
    metrics = trainer.evaluate(ds['test'])
    save_dir = f'model_{INSTITUTION}_intent_{name}'
    trainer.save_model(save_dir); tok.save_pretrained(save_dir)
    return metrics, save_dir

In [ ]:
results = {}
saved = {}
for name, ckpt in MODELS.items():
    m, d = train_one(name, ckpt)
    results[name] = m
    saved[name] = d

report = pd.DataFrame({k: {
    'accuracy': v['eval_accuracy'], 'precision': v['eval_precision'],
    'recall': v['eval_recall'], 'f1': v['eval_f1']} for k, v in results.items()}).T
report = report.sort_values('f1', ascending=False)
print(report)
report.to_csv(f'{INSTITUTION}_intent_comparison.csv')

In [ ]:
import matplotlib.pyplot as plt
ax = report[['accuracy', 'precision', 'recall', 'f1']].plot(kind='bar', figsize=(9, 5))
ax.set_title(f'{INSTITUTION.upper()} Intent Classifier Comparison')
ax.set_ylim(0, 1); ax.set_ylabel('score'); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(f'{INSTITUTION}_intent_comparison.png', dpi=120); plt.show()

In [ ]:
best = report.index[0]
print('Best model:', best, '->', saved[best])
import shutil
shutil.make_archive(f'{INSTITUTION}_intent_best', 'zip', saved[best])
print('Zipped:', f'{INSTITUTION}_intent_best.zip  (download and/or push to HF Hub)')

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # paste your HF token
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    repo = f'{HF_USER}/{INSTITUTION}-intent-{best}'
    AutoModelForSequenceClassification.from_pretrained(saved[best]).push_to_hub(repo)
    AutoTokenizer.from_pretrained(saved[best]).push_to_hub(repo)
    print('Pushed to', repo)
    print(f'Set in .env:  {INSTITUTION.upper()}_INTENT_MODEL={repo}')